In [1]:
# Core
import torch
import torch.nn as nn
import torch.optim as optim
from torch.nn import MSELoss
# Data handling
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader,TensorDataset

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Metrics & utilities
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split

# Progress & debugging
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
import glob
import joblib

device = "mps" if torch.backends.mps.is_available() else "cpu"


In [2]:
all_files = sorted(glob.glob("data/monthly_hourly_load_values_*.xlsx"))
df_list = [pd.read_excel(file) for file in all_files]
df = pd.concat(df_list, ignore_index=True)
df['DateUTC'] = pd.to_datetime(df['DateUTC'])
df.set_index('DateUTC', inplace=True)

# Training: 2019–2024, Testing: 2025
train_df_full = df[df.index.year < 2025]
test_df = df[df.index.year == 2025]

# Split validation from end of 2024 (10% of training)
val_ratio = 0.1
val_size = int(len(train_df_full) * val_ratio)
val_df = train_df_full.iloc[-val_size:]
train_df = train_df_full.iloc[:-val_size]

def create_sequences(data, seq_len=24, output_len=1):
    X, y = [], []
    for i in range(seq_len, len(data) - output_len + 1):
        X.append(data[i - seq_len:i])
        y.append(data[i:i + output_len].flatten())
    return np.array(X), np.array(y)



def prepare_multi_country_data_per_country_scaler(df, sequence_length=24, prediction_length=1, dataset_name="train/val"):
    """
    For each country, fit a StandardScaler, save it, and normalize values.
    """
    print(f"\n{'='*50}")
    print(f"PREPARING {dataset_name.upper()} WITH PER-COUNTRY SCALER")
    print(f"{'='*50}")

    df_clean = df.dropna(subset=['Value']).copy()
    df_clean = df_clean.sort_values(['CountryCode', 'DateUTC']).reset_index(drop=False)

    sequences = []
    targets = []
    countries = []
    timestamps = []

    for country in df_clean['CountryCode'].unique():
        country_data = df_clean[df_clean['CountryCode'] == country]
        scaler = StandardScaler()
        scaler.fit(country_data[['Value']])
        # Save scaler for later use
        # scaler_filename = f"scaler_{country}.pkl"
        # joblib.dump(scaler, scaler_filename)
        # Normalize values
        country_data['Value_normalized'] = scaler.transform(country_data[['Value']])
        values = country_data['Value_normalized'].values
        dates = country_data['DateUTC'].values
        X_seq, y_seq = create_sequences(values, seq_len=sequence_length, output_len=prediction_length)
        sequences.append(X_seq)
        targets.append(y_seq)
        countries.extend([country] * len(X_seq))
        timestamps.extend(dates[sequence_length:sequence_length+len(X_seq)])

    X = np.concatenate(sequences, axis=0) if sequences else np.array([])
    y = np.concatenate(targets, axis=0) if targets else np.array([])
    countries = np.array(countries)
    timestamps = np.array(timestamps)

    print(f"\nFinal dataset stats:")
    print(f"Total sequences: {len(X):,}")
    print(f"X shape: {X.shape}")
    print(f"y shape: {y.shape}")
    print(f"Normalized X range: [{X.min():.3f}, {X.max():.3f}]")
    print(f"Normalized y range: [{y.min():.3f}, {y.max():.3f}]")

    unique_countries, counts = np.unique(countries, return_counts=True)
    print(f"\nCountry distribution in sequences:")
    for country, count in zip(unique_countries, counts):
        print(f"  {country}: {count:,} sequences ({count/len(X)*100:.1f}%)")

    return X, y

X_train, y_train = prepare_multi_country_data_per_country_scaler(
    train_df, sequence_length=24, prediction_length=1
)

X_val, y_val= prepare_multi_country_data_per_country_scaler(
    val_df, sequence_length=24, prediction_length=1
)

X_test, y_test = prepare_multi_country_data_per_country_scaler(
    test_df, sequence_length=24, prediction_length=1,dataset_name="test"
)

print(f"All datasets normalized with the same scaler")
print(f"Train range: [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"Val range:   [{X_val.min():.3f}, {X_val.max():.3f}]")
print(f"Test range:  [{X_test.min():.3f}, {X_test.max():.3f}]")



PREPARING TRAIN/VAL WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 1,649,186
X shape: (1649186, 24)
y shape: (1649186, 1)
Normalized X range: [-6.300, 43.175]
Normalized y range: [-6.300, 43.175]

Country distribution in sequences:
  AL: 24,912 sequences (1.5%)
  AT: 52,584 sequences (3.2%)
  BA: 47,982 sequences (2.9%)
  BE: 52,584 sequences (3.2%)
  BG: 52,584 sequences (3.2%)
  CH: 52,583 sequences (3.2%)
  CY: 27,617 sequences (1.7%)
  CZ: 52,578 sequences (3.2%)
  DE: 52,584 sequences (3.2%)
  DK: 52,583 sequences (3.2%)
  EE: 52,577 sequences (3.2%)
  ES: 52,583 sequences (3.2%)
  FI: 52,584 sequences (3.2%)
  FR: 52,519 sequences (3.2%)
  GB: 36,816 sequences (2.2%)
  GE: 25,806 sequences (1.6%)
  GR: 43,911 sequences (2.7%)
  HR: 43,800 sequences (2.7%)
  HU: 43,798 sequences (2.7%)
  IE: 38,057 sequences (2.3%)
  IT: 43,800 sequences (2.7%)
  LT: 43,798 sequences (2.7%)
  LU: 43,800 sequences (2.7%)
  LV: 43,799 sequences (2.7%)
  MD: 34,307 sequences (2.1%)
 

In [3]:
from sklearn.metrics import mean_absolute_error, mean_squared_error


def train_model(model, X_train, y_train, X_val, y_val, epochs=50, batch_size=32, learning_rate=1e-4):
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    print(f"Using device: {device}")
    
    model.to(device)
    
    # Convert data to tensors
    if isinstance(X_train, np.ndarray):
        X_train = torch.FloatTensor(X_train)
    if isinstance(y_train, np.ndarray):
        y_train = torch.FloatTensor(y_train)
    if isinstance(X_val, np.ndarray):
        X_val = torch.FloatTensor(X_val)
    if isinstance(y_val, np.ndarray):
        y_val = torch.FloatTensor(y_val)
    
    # Create DataLoaders
    train_dataset = torch.utils.data.TensorDataset(X_train, y_train)
    val_dataset = torch.utils.data.TensorDataset(X_val, y_val)
    
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    
    # Optimizer and loss
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()
    
    # Training history
    history = {
        'train_loss': [],
        'val_loss': []
    }
    
    print("Starting training...")
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        
        # Create progress bar for training
        train_pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
        
        for batch_idx, (data, target) in enumerate(train_pbar):
            data, target = data.to(device), target.to(device)
            
            # Add channel dimension if needed (batch_size, seq_len) -> (batch_size, seq_len, 1)
            if len(data.shape) == 2:
                data = data.unsqueeze(-1)
            
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
            # Update progress bar
            train_pbar.set_postfix({
                'Loss': f'{loss.item():.6f}',
                'Avg Loss': f'{train_loss/(batch_idx+1):.6f}'
            })
        
        avg_train_loss = train_loss / len(train_loader)
        history['train_loss'].append(avg_train_loss)
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        
        val_pbar = tqdm(val_loader, desc=f'Epoch {epoch+1}/{epochs} [Val]')
        
        with torch.no_grad():
            for data, target in val_pbar:
                data, target = data.to(device), target.to(device)
                if len(data.shape) == 2:
                    data = data.unsqueeze(-1)
                
                output = model(data)
                loss = criterion(output, target)
                val_loss += loss.item()
                
                # Update validation progress bar
                val_pbar.set_postfix({
                    'Loss': f'{loss.item():.6f}',
                    'Avg Loss': f'{val_loss/(len(val_pbar)+1):.6f}'
                })
        
        avg_val_loss = val_loss / len(val_loader)
        history['val_loss'].append(avg_val_loss)
        
        # Print epoch summary
        print(f'Epoch {epoch+1}/{epochs} - Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}')
    
    print("Training completed!")
    return model,history

In [4]:
def evaluate_model(y_true, y_pred, scaler=None):
    # Inverse transform if scaler is provided
    if scaler is not None:
        y_true_inv = scaler.inverse_transform(y_true)
        y_pred_inv = scaler.inverse_transform(y_pred)
    else:
        y_true_inv = y_true
        y_pred_inv = y_pred

    rmse = np.sqrt(mean_squared_error(y_true_inv, y_pred_inv))
    mae = mean_absolute_error(y_true_inv, y_pred_inv)
    mape = np.mean(np.abs((y_true_inv - y_pred_inv) / y_true_inv)) * 100

    return rmse, mae, mape, y_true_inv, y_pred_inv

In [11]:
def test_model_per_country(model, df, sequence_length=24, prediction_length=1, batch_size=32):
    """
    Test the model for each country using its own saved StandardScaler.
    """
    device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
    model.to(device)
    model.eval()

    countries = df['CountryCode'].unique()
    results_dict = {}

    for country in countries:
        country_df = df[df['CountryCode'] == country].copy()
        scaler_path = f"scaler_{country}.pkl"
        try:
            scaler = joblib.load(scaler_path)
        except FileNotFoundError:
            print(f"Scaler for {country} not found. Skipping.")
            continue

        # Prepare sequences for this country
        X_test, y_test = prepare_multi_country_data_per_country_scaler(
            country_df, sequence_length=sequence_length, prediction_length=prediction_length, dataset_name="test"
        )
        if len(X_test) == 0:
            print(f"Skipping {country}: Not enough data for sequence_length={sequence_length}")
            continue

        # Convert to tensors
        if isinstance(X_test, np.ndarray):
            X_test = torch.FloatTensor(X_test)
        if isinstance(y_test, np.ndarray):
            y_test = torch.FloatTensor(y_test)

        test_dataset = torch.utils.data.TensorDataset(X_test, y_test)
        test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for data, target in tqdm(test_loader, desc=f'Testing {country}'):
                data, target = data.to(device), target.to(device)
                if len(data.shape) == 2:
                    data = data.unsqueeze(-1)
                output = model(data)
                all_preds.append(output.cpu().numpy())
                all_targets.append(target.cpu().numpy())

        all_preds = np.concatenate(all_preds, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)

        # Inverse transform using country scaler
        rmse, mae, mape, y_true_inv, y_pred_inv = evaluate_model(all_targets, all_preds, scaler=scaler)
        print(f"{country} - RMSE: {rmse}, MAE: {mae}, MAPE: {mape}%")

        results_dict[country] = {
            'rmse': rmse,
            'mae': mae,
            'mape': mape,
            'y_true': y_true_inv,
            'y_pred': y_pred_inv
        }

    return results_dict

In [6]:
# Calculate average metrics excluding countries with MAPE > 10%
def print_average_metrics_exclude_high_mape(results_dict, mape_threshold=10):
    filtered = [v for v in results_dict.values() if v['mape'] <= mape_threshold]
    if not filtered:
        print("No countries with MAPE below threshold.")
        return
    avg_rmse = np.mean([v['rmse'] for v in filtered])
    avg_mae = np.mean([v['mae'] for v in filtered])
    avg_mape = np.mean([v['mape'] for v in filtered])
    print(f"Average metrics for countries with MAPE ≤ {mape_threshold}%:")
    print(f"  RMSE: {avg_rmse:.2f}")
    print(f"  MAE:  {avg_mae:.2f}")
    print(f"  MAPE: {avg_mape:.2f}%")


In [7]:
class TCN(nn.Module):
    def __init__(self, input_size, output_size, num_channels, kernel_size=2, dropout=0.2):
        super(TCN, self).__init__()
        from torch.nn.utils import weight_norm
        from torch.nn import Conv1d, ReLU, Dropout, Sequential, ModuleList

        layers = []
        num_levels = len(num_channels)
        for i in range(num_levels):
            dilation_size = 2 ** i
            in_channels = input_size if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [weight_norm(Conv1d(in_channels, out_channels, kernel_size,
                                          stride=1, padding=(kernel_size-1) * dilation_size,
                                          dilation=dilation_size)),
                       ReLU(),
                       Dropout(dropout)]
        
        self.network = Sequential(*layers)
        self.linear = nn.Linear(num_channels[-1], output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = x.permute(0, 2, 1)  # Change to (batch_size, input_size, seq_len)
        y1 = self.network(x)
        y1 = y1[:, :, -1]  # Take the last time step
        out = self.linear(y1)
        return out

In [ ]:
model_tcn = TCN(input_size=1, output_size=1, num_channels=[25, 50], kernel_size=3, dropout=0.2)
model_tcn,history = train_model(
    model_tcn, X_train, y_train, X_val, y_val,
    epochs=10, batch_size=64, learning_rate=1e-4
)
result_tcn = test_model_per_country(model_tcn, test_df, sequence_length=24, prediction_length=1, batch_size=64)

Using device: mps
Starting training...


Epoch 1/20 [Val]: 100%|██████████| 2857/2857 [00:05<00:00, 501.86it/s, Loss=0.220815, Avg Loss=0.093823]


Epoch 1/20 - Train Loss: 0.109056, Val Loss: 0.093855


Epoch 2/20 [Val]: 100%|██████████| 2857/2857 [00:05<00:00, 567.39it/s, Loss=0.235809, Avg Loss=0.096939]


Epoch 2/20 - Train Loss: 0.095105, Val Loss: 0.096973


...

Epoch 19/20 - Train Loss: 0.090084, Val Loss: 0.097258


Epoch 20/20 [Val]: 100%|██████████| 2857/2857 [00:04<00:00, 612.45it/s, Loss=0.296203, Avg Loss=0.102428]


Epoch 20/20 - Train Loss: 0.089901, Val Loss: 0.102464
Training completed!

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,272
X shape: (4272, 24)
y shape: (4272, 1)
Normalized X range: [-1.645, 2.744]
Normalized y range: [-1.645, 2.744]

Country distribution in sequences:
  AL: 4,272 sequences (100.0%)


Testing AL:   0%|          | 0/67 [00:00<?, ?it/s]


AttributeError: 'TCN' object has no attribute 'predict'

In [12]:
result_tcn = test_model_per_country(model_tcn, test_df, sequence_length=24, prediction_length=1, batch_size=64)


PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,272
X shape: (4272, 24)
y shape: (4272, 1)
Normalized X range: [-1.645, 2.744]
Normalized y range: [-1.645, 2.744]

Country distribution in sequences:
  AL: 4,272 sequences (100.0%)


Testing AL: 100%|██████████| 67/67 [00:00<00:00, 385.79it/s]

AL - RMSE: 89.90617125039999, MAE: 71.6629638671875, MAPE: 8.562271118164062%



PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,320
X shape: (4320, 24)
y shape: (4320, 1)
Normalized X range: [-1.986, 2.721]
Normalized y range: [-1.986, 2.721]

Country distribution in sequences:
  AT: 4,320 sequences (100.0%)


Testing AT: 100%|██████████| 68/68 [00:00<00:00, 235.42it/s]


...

Testing SK: 100%|██████████| 68/68 [00:00<00:00, 699.41it/s]

SK - RMSE: 143.7402034705322, MAE: 113.19359588623047, MAPE: 3.7774200439453125%



PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,320
X shape: (4320, 24)
y shape: (4320, 1)
Normalized X range: [-1.846, 2.403]
Normalized y range: [-1.846, 2.403]

Country distribution in sequences:
  XK: 4,320 sequences (100.0%)


Testing XK: 100%|██████████| 68/68 [00:00<00:00, 513.08it/s]

XK - RMSE: 70.01437491910143, MAE: 55.365604400634766, MAPE: 7.670269966125488%


In [13]:
print_average_metrics_exclude_high_mape(result_tcn, mape_threshold=10)

Average metrics for countries with MAPE ≤ 10%:
  RMSE: 524.20
  MAE:  411.65
  MAPE: 4.90%


In [14]:
class NBEATs(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=128, num_layers=4):
        super(NBEATs, self).__init__()
        from torch.nn import Linear, ReLU, Sequential, ModuleList

        layers = []
        for _ in range(num_layers):
            layers += [Linear(input_size, hidden_size), ReLU()]
            input_size = hidden_size
        self.network = Sequential(*layers)
        self.linear = Linear(hidden_size, output_size)

    def forward(self, x):
        # x shape: (batch_size, seq_len, input_size)
        x = x.view(x.size(0), -1)  # Flatten
        y1 = self.network(x)
        out = self.linear(y1)
        return out

In [15]:
model_nbeats = NBEATs(input_size=24, output_size=1, hidden_size=128, num_layers=4)
model_nbeats,history = train_model(
    model_nbeats, X_train, y_train, X_val, y_val,
    epochs=10, batch_size=64, learning_rate=1e-4)
results_nbeats = test_model_per_country(model_nbeats, test_df, sequence_length=24, prediction_length=1, batch_size=64)
print_average_metrics_exclude_high_mape(results_nbeats, mape_threshold=10)

Using device: mps
Starting training...


Epoch 1/10 [Val]: 100%|██████████| 2857/2857 [00:04<00:00, 616.12it/s, Loss=0.011377, Avg Loss=0.021036]


Epoch 1/10 - Train Loss: 0.032335, Val Loss: 0.021044


Epoch 2/10 [Val]: 100%|██████████| 2857/2857 [00:04<00:00, 599.63it/s, Loss=0.009838, Avg Loss=0.019785]


Epoch 2/10 - Train Loss: 0.025238, Val Loss: 0.019792


...

SI - RMSE: 50.619253628344076, MAE: 36.85025405883789, MAPE: 2.6711387634277344%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,318
X shape: (4318, 24)
y shape: (4318, 1)
Normalized X range: [-2.258, 2.545]
Normalized y range: [-2.258, 2.545]

Country distribution in sequences:
  SK: 4,318 sequences (100.0%)


Testing SK: 100%|██████████| 68/68 [00:00<00:00, 857.57it/s]


SK - RMSE: 54.37356768946884, MAE: 41.441749572753906, MAPE: 1.383567214012146%

PREPARING TEST WITH PER-COUNTRY SCALER

Final dataset stats:
Total sequences: 4,320
X shape: (4320, 24)
y shape: (4320, 1)
Normalized X range: [-1.846, 2.403]
Normalized y range: [-1.846, 2.403]

Country distribution in sequences:
  XK: 4,320 sequences (100.0%)


Testing XK: 100%|██████████| 68/68 [00:00<00:00, 879.16it/s]

XK - RMSE: 25.28078042479541, MAE: 18.20652198791504, MAPE: 2.528573751449585%
Average metrics for countries with MAPE ≤ 10%:
  RMSE: 177.23
  MAE:  126.58
  MAPE: 1.81%


In [17]:
print_average_metrics_exclude_high_mape(results_nbeats, mape_threshold=100)

Average metrics for countries with MAPE ≤ 100%:
  RMSE: 177.23
  MAE:  126.58
  MAPE: 1.81%
